In [ ]:
print('test')

In [23]:
# 한번 더 복습
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash")

response = llm.invoke('잘 응답되는지 알려줘')
print(response.content)

[{'type': 'text', 'text': '네, 아주 잘 응답하고 있습니다! 👍\n\n메시지가 정상적으로 잘 전달되었으며, 언제든지 도와드릴 준비가 되어 있습니다. \n\n궁금한 점이 있으시거나 필요한 작업이 있다면 편하게 말씀해 주세요!', 'extras': {'signature': 'EpwRCpkRAWkUfRMbqLJC/Hoz67NGv2+sGKN1/7O5rHdsbidcGovpFuN5EMc6jVgq+tTsBj6NvnRWl4RXoXSi8J/KfPcc9Q15KmjcgxUAzlfLTWIj02Cz83v3Ve31+MBIkr/lbT+BL01acPnK+EW9OyMByt0wbM12GWFQP+7cOeHzrLEy85ozejufjF6orQ5dyelVsI2XL5ac/8TOM130ODqVtc0TiOpKKu2j9DsYsTyW1q4kb/51V+NyhN73qtFeyQEoSb4HhnZZOJnCansXhmDfwfSi84wtsi4bw/3pzY0MZUxKimnuQY3zp3q2MeyxdDwJJhx3xF+POsukW22tQqx5DoqJxYU48YbVU2teOsMaKcej3VZFRlWVs0VadoF252kcVdQATj3mL1ffYN/8L4HmPL+aaVd1AblZchJLTIOONv+HjwDVaH+zbOgVmEby5tGjzb00L9IGZxAm+4SSfAS7Ef78hrvpwX/we5MMVX7eE4XJNLcQS8cWOPPMZzO/NbtVyHQYYAJ7DasKOlYL5JaVwzJ4T8xVP5P4d9vkyePq5TAPlWGNGIDO/olnCN+rHeJEdHHewJ1YQt5RXSnkoWoK1LnaHNBPtjBUJdAkBaFv4Ivq0CztuJwSx9C8TjVccUyDrFhdIapRexDaIZtSfKJdb+uGxNEw2kua5wryytb9hmX6d649aDz4OpM+db2q1oZb7CPKpH+QTjBNZiqznTDZUkTPUX2S9VEziXIApOpmKjlhVJQNRZEoNW8YSg+kpBY+743pl4gc+CpsQ+Vdh6hWxcBtLf1RW6A9WITrNWb0juvwob70gFfvxxzFbwqQ/TII

In [14]:
import json
from langchain_core.documents import Document

# json 데이터 로드
with open("../storage/country_travel.json", "r", encoding="utf-8") as file:
    data = json.load(file)
print(data[:1])

[{'id': 1, 'name': '제주도', 'country': '한국', 'description': '아름다운 해변과 한라산, 다양한 자연경관을 즐길 수 있는 섬', 'recommendation': 99}]


In [4]:
docs = []
for row in data:
    content = f"""
        위치: {row['name']}
        나라: {row['country']}
        설명: {row['description']}
        추천수: {row['recommendation']}
    """
    ## metadata는 검색 대상은 아니지만 문서에 딸려 다니는 부가정보를 담는곳
    ## 주로 검색 결과 필터링, 출처 원본 표시, 검색 후 가공에 쓸 값 보관
    docmake = Document(
       page_content= content.strip()
    )
    docs.append(docmake)

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(
    model_name="jhgan/ko-sroberta-multitask",
    model_kwargs={"device": "cpu"},  # GPU 있으면 "cuda"
    encode_kwargs={"normalize_embeddings": True} # 벡터 비교에서 중요한 건 “방향”임
)

C:\Users\human-11\AppData\Local\Temp\ipykernel_6684\2632616324.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings
C:\Users\human-11\AppData\Local\Temp\ipykernel_6684\2632616324.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
d:\KJY\menus_project_rag\backend\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ip

In [5]:
from langchain_chroma import Chroma
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    persist_directory="../chroma_db/travel_db"
)

In [18]:
print(vectorstore._collection.count())


200


In [8]:
vectorstore = Chroma(
    persist_directory="../chroma_db/travel_db",
    embedding_function=embeddings
)

In [11]:
## 사용자 질문
query = "해양지 추천"
results = vectorstore.similarity_search(query, k=3)

for doc in results:
    print(doc.page_content)

위치: 와이키키 해변
        나라: 미국
        설명: 하와이의 푸른 바다와 서핑, 열대 분위기를 즐길 수 있는 대표 해변
        추천수: 99
위치: 와이키키 해변
        나라: 미국
        설명: 하와이의 푸른 바다와 서핑, 열대 분위기를 즐길 수 있는 대표 해변
        추천수: 99
위치: 부산 해운대
        나라: 한국
        설명: 넓은 백사장과 아름다운 바다 풍경을 자랑하는 대표적인 해변
        추천수: 95


In [25]:
def search_travel(query, k=3):
    result = vectorstore.similarity_search_with_score(query, k=k)
    output = []
    for doc, distance in result:
        output.append({
            "내용": doc.page_content,
            "거리": distance
        })
    return output

query = "한국의 해양지 추천해줘"
results = search_travel(query, 5)
for r in results:
    print('거리', r['거리'])
    print('내용', r['내용'])

거리 1.0386637449264526
내용 위치: 제주도
        나라: 한국
        설명: 아름다운 해변과 한라산, 다양한 자연경관을 즐길 수 있는 섬
        추천수: 99
거리 1.0386637449264526
내용 위치: 제주도
        나라: 한국
        설명: 아름다운 해변과 한라산, 다양한 자연경관을 즐길 수 있는 섬
        추천수: 99
거리 1.05805504322052
내용 위치: 테카포 호수
        나라: 뉴질랜드
        설명: 맑고 푸른 호수와 밤하늘의 별빛을 감상할 수 있는 아름다운 관광지
        추천수: 98
거리 1.05805504322052
내용 위치: 테카포 호수
        나라: 뉴질랜드
        설명: 맑고 푸른 호수와 밤하늘의 별빛을 감상할 수 있는 아름다운 관광지
        추천수: 98
거리 1.0587078332901
내용 위치: 부산 해운대
        나라: 한국
        설명: 넓은 백사장과 아름다운 바다 풍경을 자랑하는 대표적인 해변
        추천수: 95


In [ ]:
## 거리가 짧은 데이터 검색 함수
def retrieval_docs(query, k=5):
    results = vectorstore.similarity_search(query, k = k)
    return results
## 보강
def format_docs(docs):
    data = ''
    for doc_data in docs:
        data = data + "\n\n" + doc_data.page_content
    print(data)
    return data
query = "한국 유명한 관광지 추천해줘"
format_docs(retrieval_docs(query,5))





위치: 전주 한옥마을
        나라: 한국
        설명: 전통 한옥과 한복 체험, 다양한 한국 음식을 즐길 수 있는 관광지
        추천수: 94

위치: 전주 한옥마을
        나라: 한국
        설명: 전통 한옥과 한복 체험, 다양한 한국 음식을 즐길 수 있는 관광지
        추천수: 94

위치: 제주도
        나라: 한국
        설명: 아름다운 해변과 한라산, 다양한 자연경관을 즐길 수 있는 섬
        추천수: 99

위치: 제주도
        나라: 한국
        설명: 아름다운 해변과 한라산, 다양한 자연경관을 즐길 수 있는 섬
        추천수: 99

위치: 남산서울타워
        나라: 한국
        설명: 서울의 아름다운 전경과 화려한 야경을 감상할 수 있는 랜드마크
        추천수: 92
[{'내용': '위치: 제주도\n        나라: 한국\n        설명: 아름다운 해변과 한라산, 다양한 자연경관을 즐길 수 있는 섬\n        추천수: 99', '거리': 1.0386637449264526}, {'내용': '위치: 제주도\n        나라: 한국\n        설명: 아름다운 해변과 한라산, 다양한 자연경관을 즐길 수 있는 섬\n        추천수: 99', '거리': 1.0386637449264526}, {'내용': '위치: 테카포 호수\n        나라: 뉴질랜드\n        설명: 맑고 푸른 호수와 밤하늘의 별빛을 감상할 수 있는 아름다운 관광지\n        추천수: 98', '거리': 1.05805504322052}, {'내용': '위치: 테카포 호수\n        나라: 뉴질랜드\n        설명: 맑고 푸른 호수와 밤하늘의 별빛을 감상할 수 있는 아름다운 관광지\n        추천수: 98', '거리': 1.05805504322052}, {'내용': '위치: 부산 해운대\n        나라: 한국\n  

In [24]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser

promt = ChatPromptTemplate.from_template("""
[역할] 여행지를 추천하고 설명과 추천수를 언급하는 여행사야
[사용자질문] {query}
[참고정보] {context}
[규칙]
- 참고정보만 참고해서 답변을 만들고 새로운 여행지를 추가하지 않는다
- 설명은 3~5줄로 한다
""")

rag_chain = (
    {
        "query": lambda x : x['usequery'],
        "context": RunnableLambda(lambda x: format_docs(retrieval_docs(x['usequery'])))
    }
    |promt
    |llm
    |StrOutputParser()
)

result = rag_chain.invoke({'usequery':'한국의 해양지 추천해줘'})
print(result)



위치: 제주도
        나라: 한국
        설명: 아름다운 해변과 한라산, 다양한 자연경관을 즐길 수 있는 섬
        추천수: 99

위치: 제주도
        나라: 한국
        설명: 아름다운 해변과 한라산, 다양한 자연경관을 즐길 수 있는 섬
        추천수: 99

위치: 테카포 호수
        나라: 뉴질랜드
        설명: 맑고 푸른 호수와 밤하늘의 별빛을 감상할 수 있는 아름다운 관광지
        추천수: 98

위치: 테카포 호수
        나라: 뉴질랜드
        설명: 맑고 푸른 호수와 밤하늘의 별빛을 감상할 수 있는 아름다운 관광지
        추천수: 98

위치: 부산 해운대
        나라: 한국
        설명: 넓은 백사장과 아름다운 바다 풍경을 자랑하는 대표적인 해변
        추천수: 95
안녕하세요! 여행 전문가로서 문의하신 한국의 멋진 해양 여행지를 추천해 드립니다.

---

### 1. 제주도
* **추천수:** 99
* **설명:**
제주도는 아름다운 해변을 품고 있는 한국의 대표적인 섬입니다.
웅장한 한라산과 함께 다채롭고 뛰어난 자연경관을 마음껏 즐기실 수 있습니다.
바다와 자연이 어우러진 천혜의 환경에서 완벽한 휴식을 즐겨보세요.

---

### 2. 부산 해운대
* **추천수:** 95
* **설명:**
부산 해운대는 탁 트인 넓은 백사장을 자랑하는 대표적인 해변입니다.
시원하고 아름다운 바다 풍경이 펼쳐져 있어 해양 관광을 즐기기에 제격입니다.
매력적인 바다 전경과 함께 잊지 못할 즐거운 추억을 만들어보시길 바랍니다.
